In [2]:
import os
import cv2
import csv
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import xml.etree.ElementTree as xet
import shutil
import gdown
import zipfile
import torch
import time
from paddleocr import PaddleOCR
from ultralytics import YOLO
from glob import glob
from lxml import etree
from sklearn.model_selection import train_test_split
from PIL import Image, ExifTags
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer


In [ ]:
print("CUDA Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


In [ ]:
# === Configuration ===
yaml_path = "data/data.yaml"
save_root = "outputs/multi_model_runs"
os.makedirs(save_root, exist_ok=True)

# === Define Model Variants ===
model_variants = [
    {"name": "ModelA", "lr0": 0.01, "batch": 8},
    {"name": "ModelB", "lr0": 0.001, "batch": 8},
    {"name": "ModelC", "lr0": 0.005, "batch": 16}
]

# === Prepare CSV Log File ===
csv_path = os.path.join(save_root, "model_results.csv")
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "Model", "Epoch", "Train/Box", "Train/Cls", "Train/DFL",
        "Val/Box", "Val/Cls", "Val/DFL",
        "Precision", "Recall", "mAP50", "mAP50-95"
    ])

# === Train Models ===
for model_cfg in model_variants:
    model_name = model_cfg["name"]
    print(f"\nStarting training for {model_name}...\n")

    model = YOLO('yolov8m.pt')
    results = model.train(
        data=yaml_path,
        epochs=300,
        imgsz=640,
        batch=model_cfg["batch"],
        device=0,
        patience=50,
        lr0=model_cfg["lr0"],
        lrf=0.01,
        project=save_root,
        name=model_name,
        save_period=50,
        val=True
    )

    metrics = results.results
    for epoch_idx in range(len(metrics['metrics/mAP50'])):
        with open(csv_path, "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                model_name,
                epoch_idx + 1,
                metrics['train/box_loss'][epoch_idx],
                metrics['train/cls_loss'][epoch_idx],
                metrics['train/dfl_loss'][epoch_idx],
                metrics['val/box_loss'][epoch_idx],
                metrics['val/cls_loss'][epoch_idx],
                metrics['val/dfl_loss'][epoch_idx],
                metrics['metrics/precision(B)'][epoch_idx],
                metrics['metrics/recall(B)'][epoch_idx],
                metrics['metrics/mAP50(B)'][epoch_idx],
                metrics['metrics/mAP50-95(B)'][epoch_idx]
            ])

    print(f"Finished training {model_name}.\n")

    # === Manual Pause ===
    if model_name != model_variants[-1]["name"]:
        print("Training paused. Take a break or allow the system to cool. Press Enter to continue...")
        input()

print(f"\nTraining complete. Results logged to: {csv_path}")


In [ ]:
# === Load the trained model === 
model_path = "weights/Model/best.pt"
model = YOLO(model_path)

# === Run validation === 
results = model.val(data="data/data.yaml", split='val')

# === Print overall results === 
print("\nOverall Validation Results:")
print(f"mAP@0.5: {results.box.map50:.3f}")
print(f"mAP@0.5:0.95: {results.box.map:.3f}")

# === Per-class metrics === 
print("\n🔍 Per-Class Results:")
class_names = results.names
per_class_data = {
    "Class": [],
    "Precision": results.box.p.tolist(),
    "Recall": results.box.r.tolist(),
    "mAP50": results.box.ap50.tolist(),
    "mAP50-95": results.box.ap.tolist()
}

for i, name in class_names.items():
    per_class_data["Class"].append(name)

df = pd.DataFrame(per_class_data)
print(df)

# === Optional: Save the results table as CSV === 
df.to_csv("validation_per_class_results.csv", index=False)
print("\nResults saved to 'validation_per_class_results.csv'")

In [ ]:
# === Config Paths ===
model_path = "weights/Model/best.pt"
test_folder = "data/TEST"
save_root = "outputs"
output_csv = "outputs/ocr_results_model.csv"

# === Load Models ===
model = YOLO(model_path)
ocr_engine = PaddleOCR(use_angle_cls=True, lang='en', use_gpu=False)

# === Load Super-Resolution Model ===
sr_model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
upsampler = RealESRGANer(
    scale=4,
    model_path = "weights/Real-ESRGAN/RealESRGAN_x4plus.pth",
    model=sr_model,
    tile=0,
    tile_pad=10,
    pre_pad=0,
    half=False,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

# === Fix Image Orientation ===
def correct_rotation(image):
    try:
        for orientation in ExifTags.TAGS.keys():
            if ExifTags.TAGS[orientation] == 'Orientation':
                break
        exif = dict(image._getexif().items())
        if exif[orientation] == 3:
            image = image.rotate(180, expand=True)
        elif exif[orientation] == 6:
            image = image.rotate(270, expand=True)
        elif exif[orientation] == 8:
            image = image.rotate(90, expand=True)
    except:
        pass
    return image

# === Super-Resolution ===
def apply_super_resolution(img):
    try:
        sr_img, _ = upsampler.enhance(img, outscale=4)
        return sr_img
    except Exception as e:
        print(f"[!] Real-ESRGAN failed: {e}")
        h, w = img.shape[:2]
        return cv2.resize(img, (w * 3, h * 3), interpolation=cv2.INTER_CUBIC)

# === OCR with Confidence Filtering ===
def ocr_plate_strict(img, conf_thresh=0.6):
    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    result = ocr_engine.ocr(img, cls=True)
    if result and result[0]:
        characters_with_conf = [(line[1][0], line[1][1]) for line in result[0] if line[1][1] >= conf_thresh]
        characters = [char for char, conf in characters_with_conf]
        confidences = [conf for char, conf in characters_with_conf]
        if characters:
            text = ''.join(e for e in ' '.join(characters) if e.isalnum()).upper()
            avg_conf = np.mean(confidences)
            return text, avg_conf
    return "N/A", 0.0

# === Debug Visualization Paths ===
debug_output_dir = os.path.join(save_root, "debug_visuals")
os.makedirs(debug_output_dir, exist_ok=True)
raw_dir = os.path.join(debug_output_dir, "raw")
preproc_dir = os.path.join(debug_output_dir, "preprocessed")
ocr_dir = os.path.join(debug_output_dir, "ocr_output")
os.makedirs(raw_dir, exist_ok=True)
os.makedirs(preproc_dir, exist_ok=True)
os.makedirs(ocr_dir, exist_ok=True)

# === Process All Test Images ===
rows = []
image_paths = sorted(
    glob(os.path.join(test_folder, '*.jpg')) +
    glob(os.path.join(test_folder, '*.jpeg'))
)

for img_path in image_paths:
    try:
        pil_img = Image.open(img_path)
        pil_img = correct_rotation(pil_img)
        img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        results = model(img)
        boxes = results[0].boxes.xyxy.cpu().numpy()
        confs = results[0].boxes.conf.cpu().numpy()

        if len(boxes) == 0:
            rows.append([os.path.basename(img_path), 'No Detection'])
            continue

        top_k = min(2, len(boxes))
        sorted_indices = confs.argsort()[-top_k:][::-1]

        texts = []
        for i, idx in enumerate(sorted_indices):
            x1, y1, x2, y2 = map(int, boxes[idx])
            roi = img[y1:y2, x1:x2]

            # === Save raw === 
            raw_path = os.path.join(raw_dir, f"{os.path.basename(img_path).split('.')[0]}_roi{i}.jpg")
            cv2.imwrite(raw_path, roi)

            # === Super-resolution === 
            sr_img = apply_super_resolution(roi)
            sr_path = os.path.join(preproc_dir, f"{os.path.basename(img_path).split('.')[0]}_sr{i}.jpg")
            cv2.imwrite(sr_path, sr_img)

            # === OCR directly on SR image === 
            text, confidence = ocr_plate_strict(sr_img)

            if text != "N/A":
                texts.append((text, confidence))

                # === Save OCR overlay image === 
                overlay = sr_img.copy()
                if len(overlay.shape) == 2:
                    overlay = cv2.cvtColor(overlay, cv2.COLOR_GRAY2BGR)
                cv2.putText(overlay, f"{text} ({round(confidence * 100)}%)", (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                ocr_path = os.path.join(ocr_dir, f"{os.path.basename(img_path).split('.')[0]}_ocr{i}.jpg")
                cv2.imwrite(ocr_path, overlay)

        final_text, final_conf = texts[0] if texts else ("N/A", 0.0)
        rows.append([os.path.basename(img_path), final_text, round(final_conf, 4)])

    except Exception as e:
        print(f"❌ Error processing {img_path}: {str(e)}")
        rows.append([os.path.basename(img_path), f'Error: {str(e)}'])

# === Save Output CSV === 
with open(output_csv, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['Image', 'LicensePlateText', 'Confidence'])
    writer.writerows(rows)

print(f"\nOCR results saved to: {output_csv}")
print(f"Debug images saved to: {debug_output_dir}")

In [ ]:
# === Evaluate Against Ground Truth ===
pred_csv = output_csv
gt_csv = "data/ground_truth.csv"
output_eval_csv = "outputs/accuracy_eval.csv"

pred_df = pd.read_csv(pred_csv)
gt_df = pd.read_csv(gt_csv)
merged = pred_df.merge(gt_df, on='Image', how='left')

def normalize_plate(text):
    if pd.isna(text): return "N/A"
    return ''.join(e for e in str(text) if e.isalnum()).upper()

merged['NormalizedPred'] = merged['LicensePlateText'].apply(normalize_plate)
merged['NormalizedGT'] = merged['GroundTruth'].apply(normalize_plate)
merged['Correct'] = merged.apply(lambda row: "Yes" if row['NormalizedPred'] == row['NormalizedGT'] else "No", axis=1)

merged[['Image', 'LicensePlateText', 'Confidence', 'GroundTruth', 'Correct']].to_csv(output_eval_csv, index=False)
print(f"\nEvaluation complete! Saved to: {output_eval_csv}")